# theLook Analytics · 02 · dbt e MetricFlow

Questo notebook ricostruisce staging e mart con **dbt** a partire dalla cartella `dbt/` del repository, esegue tutti i test (compreso il controllo anomalie sui ricavi settimanali) e interroga le metriche del **semantic layer** con **MetricFlow**. Alla fine confronta i numeri con il report Power BI: devono coincidere, perché le definizioni sono le stesse.

- dbt scrive nei dataset `dbt_staging` e `dbt_marts`, separati dalle viste create a mano negli script SQL.
- Nessuna installazione sul computer: tutto gira in Google Colab, con lo stesso account Google del progetto BigQuery.
- Serve che il repository sia pubblico su GitHub.

**Come eseguirlo**: scrivi l'ID del progetto e l'indirizzo del repository nella prima cella, poi *Runtime → Esegui tutto*.

In [ ]:
PROJECT_ID = "scrivi-qui-l-id-del-progetto"
GITHUB_REPO = "https://github.com/TUO-UTENTE/thelook-analytics.git"

if PROJECT_ID.startswith("scrivi-qui"):
    raise ValueError("Scrivi l'ID del progetto Google Cloud in PROJECT_ID, poi riesegui il notebook.")
if "TUO-UTENTE" in GITHUB_REPO:
    raise ValueError("Scrivi l'indirizzo del tuo repository GitHub in GITHUB_REPO, poi riesegui il notebook.")

In [ ]:
# Accesso a BigQuery con l'account Google: dbt e MetricFlow useranno queste credenziali
from google.colab import auth
auth.authenticate_user()

## 1 · Installazione

dbt e MetricFlow vanno in un ambiente Python 3.12 separato, creato con `uv`, così non dipendono dalla versione di Python di Colab. Ci vuole circa un minuto.

In [ ]:
import os

!pip install -q uv
!uv venv -q --python 3.12 /content/dbt-env
!uv pip install -q --python /content/dbt-env/bin/python "dbt-metricflow[dbt-bigquery]"

os.environ["PATH"] = "/content/dbt-env/bin:" + os.environ["PATH"]
!dbt --version

In [ ]:
import shutil

# Copia aggiornata del repository
if os.path.exists("/content/repo"):
    shutil.rmtree("/content/repo")
!git clone -q {GITHUB_REPO} /content/repo
%cd /content/repo/dbt

# Profilo di connessione: OAuth con le credenziali di Colab, location US come i dati pubblici
profilo = f"""thelook:
  target: dev
  outputs:
    dev:
      type: bigquery
      method: oauth
      project: {PROJECT_ID}
      dataset: dbt
      location: US
      threads: 4
"""
# Il profilo va nella cartella del progetto: dbt lo trova lì, e MetricFlow
# lo cerca solo lì (o in DBT_PROFILES_DIR)
with open("profiles.yml", "w") as f:
    f.write(profilo)
os.environ["DBT_PROFILES_DIR"] = os.getcwd()

!dbt debug

## 2 · Build e test

`dbt build` carica il seed dei canali, crea i modelli nell'ordine giusto e dopo ciascuno esegue i suoi test. Nel riepilogo finale:

- **PASS**: test superato.
- **WARN**: avviso che non blocca, per esempio il controllo anomalie sulle settimane o uno stato d'ordine inatteso.
- **ERROR** o **FAIL**: va guardato prima di proseguire.

In [ ]:
!dbt build

### Dettaglio di errori e avvisi

Per ogni test non superato, questa cella rilancia la query del test e mostra le righe che lo hanno fatto scattare. Per il controllo anomalie sono le settimane segnalate, con lo scarto rispetto alla media delle quattro precedenti.

In [ ]:
import json
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID)
risultati = json.load(open("target/run_results.json"))["results"]
segnalati = [r for r in risultati if r["status"] in ("fail", "warn", "error")]

if not segnalati:
    print("Nessun errore e nessun avviso.")
for r in segnalati:
    print(f"\n{r['status'].upper()} · {r['unique_id'].split('.')[2]} · {r.get('failures')} righe")
    sql = r.get("compiled_code")
    if sql:
        display(bq.query(sql).to_dataframe(create_bqstorage_client=False).head(20))

## 3 · Semantic layer

`mf validate-configs` controlla che semantic model e metriche siano coerenti tra loro e con i dati in BigQuery.

In [ ]:
!mf validate-configs

## 4 · Confronto con Power BI

Stesso periodo del report, gennaio–settembre 2026. I valori attesi, letti nelle schede di Power BI, sono indicati sopra ogni query.

In [ ]:
# Atteso da Power BI: ricavi netti 2,20 milioni · ordini 29K · valore medio 85,83 · margine 51,9% · resi 11,7%
!mf query --metrics ricavi_netti,ordini,valore_medio_ordine,margine_pct,tasso_di_reso --group-by metric_time__year --start-time 2026-01-01 --end-time 2026-09-30

In [ ]:
# Atteso da Power BI: acquirenti 24K · nuovi 17K · di ritorno 10K · quota di ritorno 41,8%
!mf query --metrics clienti_acquirenti,nuovi_clienti,clienti_di_ritorno,quota_clienti_di_ritorno --group-by metric_time__year --start-time 2026-01-01 --end-time 2026-09-30

In [ ]:
# Atteso da Power BI: Outerwear & Coats 273K · Jeans 251K · Sweaters 169K
!mf query --metrics ricavi_netti --group-by product__category --start-time 2026-01-01 --end-time 2026-09-30 --order -ricavi_netti --limit 5

### Lettura dei risultati

*Da completare dopo l'esecuzione.*